# 🤖 Chương 6: Machine Learning cho Time Series
## Advanced Data Science - Session 5

---

**Mục tiêu chương này:**
- Biến Time Series thành bài toán Supervised Learning
- Feature Engineering: lag, rolling, date features
- Áp dụng Random Forest, XGBoost, LightGBM
- TimeSeriesSplit Cross-Validation
- So sánh ML vs Traditional Methods

## 6.1 Từ Time Series → Supervised Learning

### 🎯 Ý tưởng cốt lõi:
Time Series: Y₁, Y₂, Y₃, ..., Yₙ → Dự đoán Y_{n+1}

**Biến đổi thành bảng features:**

| Y_{t-3} | Y_{t-2} | Y_{t-1} | → Y_t |
|---------|---------|---------|-------|
| 10 | 12 | 15 | **18** |
| 12 | 15 | 18 | **20** |
| 15 | 18 | 20 | **22** |

→ Bây giờ có X (features) và y (target) → Supervised Learning!

### 6.2 Các Loại Features

| Loại | Ví dụ | Ý nghĩa |
|------|-------|----------|
| **Lag features** | Y_{t-1}, Y_{t-7} | Giá trị quá khứ |
| **Rolling stats** | Mean/Std 7 ngày | Xu hướng ngắn hạn |
| **Date features** | Tháng, thứ, quý | Pattern theo thời gian |
| **Expanding** | Expanding mean | Trung bình tích lũy |
| **Diff features** | Y_t - Y_{t-1} | Tốc độ thay đổi |

### 6.3 TimeSeriesSplit
⚠️ **KHÔNG dùng KFold thông thường!** Vì:
- KFold: Trộn lẫn past & future → Data Leakage!
- TimeSeriesSplit: Luôn train trên past, test trên future

```
Fold 1: [Train    ] [Test]
Fold 2: [Train         ] [Test]
Fold 3: [Train              ] [Test]
Fold 4: [Train                   ] [Test]
```

---
## 🔬 Phần Thực Hành
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

def evaluate(actual, predicted, model_name=''):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    print(f'📊 {model_name:25s} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

print('✅ Import thành công!')

### 📝 Ví dụ 1: Feature Engineering Hoàn Chỉnh

In [ ]:
# Tạo dữ liệu mẫu
np.random.seed(42)
n = 730  # 2 năm
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)

trend = 0.02 * t
seasonal = 10 * np.sin(2 * np.pi * t / 365) + 5 * np.sin(2 * np.pi * t / 7)
noise = np.random.normal(0, 3, n)
sales = 100 + trend + seasonal + noise

df = pd.DataFrame({'date': dates, 'sales': sales})
df.set_index('date', inplace=True)

print('🔧 Feature Engineering Pipeline:')
print('='*50)

def create_features(df, target_col='sales', lags=[1,2,3,7,14,30]):
    """Tạo features từ time series"""
    data = df.copy()
    
    # 1. LAG FEATURES
    print('\n📌 1. Lag Features:')
    for lag in lags:
        data[f'lag_{lag}'] = data[target_col].shift(lag)
        print(f'   lag_{lag}: Giá trị {lag} ngày trước')
    
    # 2. ROLLING FEATURES
    print('\n📌 2. Rolling Features:')
    windows = [7, 14, 30]
    for w in windows:
        data[f'rolling_mean_{w}'] = data[target_col].shift(1).rolling(w).mean()
        data[f'rolling_std_{w}'] = data[target_col].shift(1).rolling(w).std()
        print(f'   rolling_mean_{w}: TB trượt {w} ngày')
        print(f'   rolling_std_{w}: Độ lệch chuẩn {w} ngày')
    
    # 3. DATE FEATURES
    print('\n📌 3. Date Features:')
    data['day_of_week'] = data.index.dayofweek
    data['day_of_month'] = data.index.day
    data['month'] = data.index.month
    data['quarter'] = data.index.quarter
    data['is_weekend'] = (data.index.dayofweek >= 5).astype(int)
    data['day_of_year'] = data.index.dayofyear
    print('   day_of_week, day_of_month, month, quarter, is_weekend, day_of_year')
    
    # 4. DIFF FEATURES
    print('\n📌 4. Diff Features:')
    data['diff_1'] = data[target_col].diff(1)
    data['diff_7'] = data[target_col].diff(7)
    print('   diff_1: Thay đổi so với hôm qua')
    print('   diff_7: Thay đổi so với tuần trước')
    
    # 5. EXPANDING FEATURES
    print('\n📌 5. Expanding Features:')
    data['expanding_mean'] = data[target_col].shift(1).expanding().mean()
    print('   expanding_mean: Trung bình tích lũy')
    
    # Drop NaN
    data = data.dropna()
    print(f'\n✅ Tổng features: {len(data.columns) - 1}')
    print(f'✅ Samples sau khi drop NaN: {len(data)}')
    
    return data

df_featured = create_features(df)
df_featured.head()

### 📝 Ví dụ 2: TimeSeriesSplit vs KFold

In [ ]:
from sklearn.model_selection import KFold

# Minh họa sự khác biệt
n_samples = 100
X_demo = np.arange(n_samples)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# KFold (SAI cho Time Series)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(kf.split(X_demo)):
    axes[0].scatter(train_idx, [i]*len(train_idx), c='blue', marker='|', s=100)
    axes[0].scatter(test_idx, [i]*len(test_idx), c='red', marker='|', s=100)
axes[0].set_title('❌ KFold: Trộn lẫn past & future → DATA LEAKAGE!', fontweight='bold')
axes[0].set_ylabel('Fold')

# TimeSeriesSplit (ĐÚNG)
tscv = TimeSeriesSplit(n_splits=5)
for i, (train_idx, test_idx) in enumerate(tscv.split(X_demo)):
    axes[1].scatter(train_idx, [i]*len(train_idx), c='blue', marker='|', s=100)
    axes[1].scatter(test_idx, [i]*len(test_idx), c='red', marker='|', s=100)
axes[1].set_title('✅ TimeSeriesSplit: Train trên past, test trên future → ĐÚNG!', fontweight='bold')
axes[1].set_ylabel('Fold')
axes[1].set_xlabel('Time Index')

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='|', color='blue', markersize=15, label='Train'),
                   Line2D([0], [0], marker='|', color='red', markersize=15, label='Test')]
axes[0].legend(handles=legend_elements, loc='upper right')
axes[1].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print('💡 KFold trộn past & future → Model "nhìn trước" tương lai → Data Leakage!')
print('💡 TimeSeriesSplit luôn đảm bảo: Train trước Test theo thời gian')

### 📝 Ví dụ 3: Random Forest cho Time Series

In [ ]:
# Prepare data
target = 'sales'
feature_cols = [c for c in df_featured.columns if c != target]

# Train/Test split (80/20) - PHẢI THEO THỜI GIAN!
split_idx = int(len(df_featured) * 0.8)
X_train = df_featured[feature_cols][:split_idx]
y_train = df_featured[target][:split_idx]
X_test = df_featured[feature_cols][split_idx:]
y_test = df_featured[target][split_idx:]

print(f'Train: {X_train.shape[0]} samples ({X_train.index[0].date()} → {X_train.index[-1].date()})')
print(f'Test:  {X_test.shape[0]} samples ({X_test.index[0].date()} → {X_test.index[-1].date()})')

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print('\n📊 Random Forest Results:')
evaluate(y_test, rf_pred, 'Random Forest')

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
top_15 = importances.tail(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot predictions
axes[0].plot(y_test.index, y_test, label='Actual', color='black', linewidth=2)
axes[0].plot(y_test.index, rf_pred, label='Random Forest', color='green', alpha=0.8)
axes[0].legend()
axes[0].set_title('📊 Random Forest Forecast', fontweight='bold')

# Feature importance
top_15.plot(kind='barh', ax=axes[1], color='forestgreen')
axes[1].set_title('📊 Top 15 Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print('\n💡 Lag_1 thường là feature quan trọng nhất (giá trị gần nhất)')
print('💡 Date features giúp bắt seasonality')

### 📝 Ví dụ 4: So Sánh Nhiều ML Models

In [ ]:
# So sánh nhiều models
print('='*70)
print('  SO SÁNH CÁC ML MODELS CHO TIME SERIES')
print('='*70)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
}

# Try import XGBoost/LightGBM
try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(n_estimators=200, max_depth=5, random_state=42, verbosity=0)
except ImportError:
    print('⚠️ XGBoost not installed, skipping...')

try:
    from lightgbm import LGBMRegressor
    models['LightGBM'] = LGBMRegressor(n_estimators=200, max_depth=5, random_state=42, verbose=-1)
except ImportError:
    print('⚠️ LightGBM not installed, skipping...')

results = {}
predictions = {}

print('\n📊 Results:')
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = evaluate(y_test, pred, name)
    predictions[name] = pred

# Best model
best = min(results.items(), key=lambda x: x[1]['RMSE'])
print(f'\n🏆 Best Model: {best[0]} (RMSE = {best[1]["RMSE"]:.4f})')

# Plot all
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test.index, y_test, label='Actual', color='black', linewidth=2)

colors = ['blue', 'green', 'orange', 'red', 'purple']
for (name, pred), color in zip(predictions.items(), colors):
    ax.plot(y_test.index, pred, label=name, color=color, alpha=0.7)

ax.legend()
ax.set_title('📊 So Sánh Các ML Models', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 📝 Ví dụ 5: TimeSeriesSplit Cross-Validation

In [ ]:
# Cross-validation đúng cách cho Time Series
print('='*60)
print('  TIMESERIES CROSS-VALIDATION')
print('='*60)

X_all = df_featured[feature_cols]
y_all = df_featured[target]

tscv = TimeSeriesSplit(n_splits=5)

best_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)

fold_scores = []

print('\n📊 Cross-Validation Results:')
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all), 1):
    X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
    y_tr, y_te = y_all.iloc[train_idx], y_all.iloc[test_idx]
    
    best_model.fit(X_tr, y_tr)
    pred = best_model.predict(X_te)
    
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae = mean_absolute_error(y_te, pred)
    fold_scores.append({'Fold': fold, 'RMSE': rmse, 'MAE': mae, 
                       'Train_size': len(train_idx), 'Test_size': len(test_idx)})
    
    print(f'  Fold {fold}: Train={len(train_idx):4d} | Test={len(test_idx):4d} | '
          f'RMSE={rmse:.4f} | MAE={mae:.4f}')

scores_df = pd.DataFrame(fold_scores)
print(f'\n  ► Average RMSE: {scores_df["RMSE"].mean():.4f} ± {scores_df["RMSE"].std():.4f}')
print(f'  ► Average MAE:  {scores_df["MAE"].mean():.4f} ± {scores_df["MAE"].std():.4f}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
x = range(1, 6)
ax.bar(x, scores_df['RMSE'], color='steelblue', alpha=0.7, label='RMSE')
ax.axhline(y=scores_df['RMSE'].mean(), color='red', linestyle='--', label=f'Mean RMSE={scores_df["RMSE"].mean():.4f}')
ax.set_xlabel('Fold')
ax.set_ylabel('RMSE')
ax.set_title('📊 TimeSeriesSplit Cross-Validation Scores', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('\n💡 CV giúp đánh giá model ổn định hơn 1 lần train/test duy nhất')
print('💡 Std nhỏ → Model ổn định')
print('💡 Std lớn → Model không ổn định, cần xem lại')

---
## 🏋️ BÀI TẬP THỰC HÀNH
---

### Bài 1: Feature Engineering (⭐ Dễ)

Với dữ liệu `retail_sales_dataset.csv`:
1. Tạo lag features (1, 7, 14, 30 ngày)
2. Tạo rolling features (7, 14, 30 ngày)
3. Tạo date features (day_of_week, month, quarter)
4. Hiển thị correlation matrix giữa features với target

In [ ]:
# TODO: Viết code ở đây

### Bài 2: ML Model Comparison (⭐⭐ Trung bình)

1. Dùng features đã tạo ở Bài 1
2. Train: Linear Regression, Random Forest, Gradient Boosting
3. Đánh giá với TimeSeriesSplit (5 folds)
4. Vẽ biểu đồ so sánh RMSE của từng fold
5. Feature importance của best model

In [ ]:
# TODO: Viết code ở đây

### Bài 3: ML vs ARIMA (⭐⭐⭐ Nâng cao)

So sánh toàn diện ML vs Classical:
1. Fit ARIMA/SARIMA (từ Chương 5)
2. Fit Random Forest + XGBoost/LightGBM
3. So sánh MAE, RMSE, MAPE
4. Phân tích: Model nào tốt hơn ở đoạn nào? Tại sao?
5. Vẽ biểu đồ so sánh predictions

In [ ]:
# TODO: Viết code ở đây

---
# 🧠 PHẦN ÔN TẬP & CỦNG CỐ KIẾN THỨC - CHƯƠNG 6

---

## 1️⃣ Nguyên Lí 80/20: 20% Kiến Thức Cốt Lõi Mang Lại 80% Giá Trị

> **Nếu bạn chỉ nhớ được 3 điều từ chương này, hãy nhớ:**

### 🔑 Kiến thức #1: Biến Time Series → Supervised Learning bằng Lag Features
```python
# Biến TS: Y1, Y2, Y3, Y4, Y5
# Thành bảng (với lag=2):
#   X1    X2   →  y
#   Y1    Y2      Y3
#   Y2    Y3      Y4
#   Y3    Y4      Y5
```
- **Ý tưởng cốt lõi:** Dùng giá trị **quá khứ** (lag) làm features, giá trị **hiện tại** làm target
- **Feature Engineering** quyết định **80%** hiệu quả ML cho Time Series
- Các loại features: Lag, Rolling (mean/std), Date (month/day_of_week), Expanding

### 🔑 Kiến thức #2: TimeSeriesSplit — KHÔNG BAO GIỜ dùng KFold!
```python
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
# Fold 1: [Train    ] [Test]
# Fold 2: [Train         ] [Test]
# Fold 3: [Train              ] [Test]
```
- **KFold trộn lẫn past & future → Data Leakage → Kết quả ảo!**
- **TimeSeriesSplit luôn train trên past, test trên future → Mô phỏng thực tế**

### 🔑 Kiến thức #3: Tree-based Models (RF, XGBoost, LightGBM) mạnh nhất cho TS
- **Random Forest:** Ổn định, ít overfit, tốt cho baseline
- **XGBoost/LightGBM:** Mạnh nhất, nhưng cần tune hyperparameters
- **Feature Importance:** Cho biết features nào quan trọng nhất → Insight về data!

> **Một câu tóm tắt:** *"Tạo features từ quá khứ (lag, rolling, date) → Train model (RF/XGBoost) → Validate bằng TimeSeriesSplit (KHÔNG KFold!)"*

---

## 2️⃣ Mô Hình Hóa Kiến Thức: Phép Ẩn Dụ "Thám Tử Phá Án"

### 🔍 Câu chuyện: ML cho Time Series giống THÁM TỬ PHÂN TÍCH VỤ ÁN

Bạn là **thám tử** cần dự đoán vụ trộm tiếp theo xảy ra khi nào:

| Thám tử | → ML cho Time Series |
|---|---|
| Thu thập **bằng chứng** từ các vụ trước | Tạo **Lag Features** từ dữ liệu quá khứ |
| Phân tích **xu hướng** (tần suất tăng? giảm?) | **Rolling Statistics** (mean, std) |
| Xem **thời điểm** (cuối tuần? ban đêm?) | **Date Features** (day_of_week, hour) |
| Tổng hợp thành **hồ sơ** | **Feature Matrix** (X) |
| Dùng kinh nghiệm để **dự đoán** | **Model** (RF, XGBoost) predict |
| Kiểm tra **trình tự** vụ án (không nhìn tương lai!) | **TimeSeriesSplit** (train trên past) |
| Đánh giá dựa trên **vụ án mới** | **Test set** = dữ liệu tương lai |

**Tại sao KHÔNG dùng KFold?**
- Giống như thám tử **nhìn trộm hồ sơ tương lai** rồi nói "Tôi đoán đúng!" → **Gian lận!**
- TimeSeriesSplit = Thám tử chỉ dùng **thông tin quá khứ** → **Công bằng!**

### 🎨 Sơ đồ Feature Engineering:
```
  Raw Time Series: Y1, Y2, Y3, ..., Yn
              │
  ┌───────────┼───────────┬──────────────┐
  │           │           │              │
  ▼           ▼           ▼              ▼
 LAG       ROLLING      DATE         EXPANDING
 Features  Features     Features     Features
  │           │           │              │
 Y(t-1)    Mean_7      Month         Cum_Mean
 Y(t-7)    Std_30      DayOfWeek    Cum_Std
 Y(t-14)   Min_7       Quarter
 Y(t-30)   Max_7       IsWeekend
  │           │           │              │
  └───────────┼───────────┴──────────────┘
              │
         FEATURE MATRIX (X)
              │
              ▼
    ┌─────────────────────┐
    │ TimeSeriesSplit CV   │
    │    ┌──────┬────┐    │
    │ F1 │Train │Test│    │
    │ F2 │Train   │Test│  │
    │ F3 │Train     │Test││
    └─────────────────────┘
              │
    ┌─────────┼─────────┐
    │         │         │
    RF     XGBoost   LightGBM
    │         │         │
    └─────────┼─────────┘
              │
         Best Model + Feature Importance
```

---

## 3️⃣ Liên Tưởng Với Cuộc Sống

### 🛒 Dự đoán lượng khách siêu thị
- **Lag Features:** "Tuần trước thứ 7 có 500 khách → Thứ 7 này cũng khoảng 500"
- **Rolling Mean:** "Trung bình 4 tuần qua là 450 khách/thứ 7 → Xu hướng ổn định"
- **Date Features:** "Cuối tuần đông hơn ngày thường, tháng 12 đông nhất"
- **TimeSeriesSplit:** "Đánh giá model bằng dữ liệu tuần sau, KHÔNG phải tuần trước"

### 📚 Dự đoán điểm thi tiếp theo
- **Lag Features:** Điểm 3 bài thi gần nhất → Dự đoán bài tiếp theo
- **Rolling Features:** Trung bình điểm 5 bài gần nhất → Năng lực hiện tại
- **Date Features:** Thi giữa kỳ vs cuối kỳ, thi sáng vs chiều
- **Model:** Random Forest cho biết "điểm bài trước" quantrọng nhất (feature importance)

### 🏠 Dự đoán giá nhà
- **Lag:** Giá bán tháng trước ảnh hưởng giá tháng này
- **Rolling:** Trung bình giá 6 tháng qua → Xu hướng thị trường
- **Date:** Cuối năm giao dịch nhiều hơn, mùa Tết ít giao dịch
- **XGBoost thường thắng** trong bài toán này trên Kaggle!

### 🎮 Dự đoán lượng người chơi game online
- **Lag Features:** Hôm qua 10,000 người → Hôm nay có thể ~10,000
- **Rolling Features:** 7 ngày qua trung bình 9,500 → Trend giảm nhẹ
- **Date Features:** Cuối tuần đông gấp đôi, nghỉ hè đông gấp 3
- **KFold sẽ sai:** Vì nhìn data mùa hè để đoán mùa xuân → Không hợp lý!

---

## 4️⃣ Teach to Learn — Giảng Lại Để Hiểu Sâu

### 📝 Bài tập 1: Giải thích Feature Engineering cho người mới
> Bạn được yêu cầu dự đoán **doanh số quán trà sữa** tuần tới.
> Liệt kê **5 features** bạn sẽ tạo và giải thích tại sao mỗi feature quan trọng.

In [ ]:
# ✍️ BÀI TẬP 1: Giải thích Feature Engineering cho người mới
# Bạn cần dự đoán doanh số quán trà sữa tuần tới

"""
5 FEATURES TÔI SẼ TẠO:

Feature 1: _______________
  → Tại sao quan trọng: _______________

Feature 2: _______________
  → Tại sao quan trọng: _______________

Feature 3: _______________
  → Tại sao quan trọng: _______________

Feature 4: _______________
  → Tại sao quan trọng: _______________

Feature 5: _______________
  → Tại sao quan trọng: _______________

TẠI SAO KHÔNG ĐƯA THẲNG DATA VÀO MODEL?
→ _______________
"""

In [ ]:
# ✍️ BÀI TẬP 2: Giải thích tại sao KHÔNG dùng KFold cho Time Series
# Vẽ sơ đồ so sánh KFold vs TimeSeriesSplit

"""
GIẢI THÍCH CHO BẠN CÙNG LỚP:

KFold sai ở đâu khi áp dụng cho TS?
→ _______________

Data Leakage là gì? Ví dụ cụ thể:
→ _______________

SƠ ĐỒ SO SÁNH:

KFold (SAI cho TS):
Fold 1: _______________
Fold 2: _______________
Fold 3: _______________

TimeSeriesSplit (ĐÚNG):
Fold 1: [Train    ] [Test]
Fold 2: [Train         ] [Test]
Fold 3: [Train              ] [Test]

Hậu quả của Data Leakage:
→ _______________
"""

In [ ]:
# ✍️ BÀI TẬP 3: ML vs ARIMA — Khi nào dùng gì?
# Viết bảng so sánh + 2 tình huống thực tế

"""
BẢNG SO SÁNH:

| Tiêu chí           | ARIMA/SARIMA      | ML (RF, XGBoost)   |
|--------------------|-------------------|---------------------|
| Khi nào tốt hơn?   | _______________   | _______________     |
| Ưu điểm            | _______________   | _______________     |
| Nhược điểm         | _______________   | _______________     |
| Cần gì từ data?    | _______________   | _______________     |
| Interpretability    | _______________   | _______________     |
| Handling features   | _______________   | _______________     |

TÌNH HUỐNG NÊN DÙNG ARIMA:
→ _______________
→ Tại sao ARIMA tốt hơn ML ở đây? _______________

TÌNH HUỐNG NÊN DÙNG ML:
→ _______________
→ Tại sao ML tốt hơn ARIMA ở đây? _______________
"""